# 02 — Test Module 1: Asymmetric Spectral Normalization

Test that Module 1 produces sensible output on a real dataset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
PROJECT_ROOT = '/content/drive/MyDrive/Project_GraphML/ms-zerogad'
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

In [ ]:
import torch
import matplotlib.pyplot as plt

from ms_zerogad.data.loader import load_graph_dataset
from ms_zerogad.data.preprocessing import sparse_to_torch_dense, feature_to_torch
from ms_zerogad.modules.unification import GlobalUnification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Load Cora

In [ ]:
A_sp, X_sp, y_np = load_graph_dataset('/content/drive/MyDrive/Project_GraphML/ms-zerogad/ms_zerogad/data/raw/Cora.mat')
A = sparse_to_torch_dense(A_sp).to(device)
X = feature_to_torch(X_sp, dense=True).to(device)

print(f'Cora: n={X.shape[0]}, f={X.shape[1]}')

## Apply Module 1

In [ ]:
module1 = GlobalUnification(
    d_prime=8,
    band_low=0.5,
    band_high=1.5,
    alpha_low=1.0,
    alpha_mid=0.95,
    alpha_high=0.9,
).to(device)

import time
t0 = time.time()
X_unified, components = module1(X, A, return_components=True)
elapsed = time.time() - t0

print(f'Time: {elapsed:.2f}s')
print(f'Output shape: {X_unified.shape}')
print(f'Output value range: [{X_unified.min().item():.4f}, {X_unified.max().item():.4f}]')

## Inspect band statistics

In [ ]:
band_info = module1.get_band_info(components['eigenvalues'])
print(f'Eigenvalue range: [{components["eigenvalues"].min().item():.4f}, {components["eigenvalues"].max().item():.4f}]')
print()
print(f'Low band  (λ < 0.5):     {band_info["low"]:5d} ({band_info["low_pct"]:.1f}%)')
print(f'Mid band  (0.5 ≤ λ < 1.5): {band_info["mid"]:5d} ({band_info["mid_pct"]:.1f}%)')
print(f'High band (λ ≥ 1.5):     {band_info["high"]:5d} ({band_info["high_pct"]:.1f}%)')

## Visualize spectrum and band split

In [ ]:
eigenvalues = components['eigenvalues'].cpu().numpy()

fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(eigenvalues, bins=80, color='steelblue', alpha=0.7)
ax.axvline(0.5, color='green', linestyle='--', label='band_low=0.5')
ax.axvline(1.5, color='red', linestyle='--', label='band_high=1.5')
ax.set_xlabel('Eigenvalue λ')
ax.set_ylabel('Count')
ax.set_title('Eigenvalue Distribution of Cora Normalized Laplacian')
ax.legend()
plt.show()

## Check that low-band features are more affected than high-band

In [ ]:
F_before = components['F_before']
F_after = components['F_after']

low_mask = components['low_mask']
mid_mask = components['mid_mask']
high_mask = components['high_mask']

print('Per-band statistics (averaged over feature dim):')
print(f'{"Band":<6} {"|F_before|":<12} {"|F_after|":<12} {"Change":<10}')
print('-' * 40)
for band, mask in [('Low', low_mask), ('Mid', mid_mask), ('High', high_mask)]:
    before = F_before[mask].abs().mean().item()
    after = F_after[mask].abs().mean().item()
    print(f'{band:<6} {before:<12.4f} {after:<12.4f} {after - before:+.4f}')